# Watching the graph grow, and a question travel through it

The real code is in `backend/src/cricketmind_v3/application/graph/`. This notebook only imports it and watches, so it never drifts from the code. Run it again each day: as the stubs become real nodes, every step shows more.

**Kernel:** Select Kernel → Python Environments → the one ending in `backend\.venv\Scripts\python.exe` (or Enter interpreter path: `D:\Cricket\cricketmind-v3-work\cricketmind-v3\backend\.venv\Scripts\python.exe`). The pictures need the VS Code extension *Markdown Preview Mermaid Support*.

In [ ]:
import sys

print(sys.executable)  # should end with backend\.venv\Scripts\python.exe

In [ ]:
from IPython.display import Markdown, display

from cricketmind_v3.application.graph import nodes
from cricketmind_v3.application.graph.build import build_graph
from cricketmind_v3.application.graph.context import GraphContext
from cricketmind_v3.domain.faith.gazetteer import cricket_gazetteer

graph = build_graph()

# what the nodes are given: the names N3 knows (a few, for this notebook)
CONTEXT = GraphContext(
    names=cricket_gazetteer(
        {
            "V Kohli": ("Kohli", "Virat Kohli"),
            "JJ Bumrah": ("Bumrah", "Jasprit Bumrah"),
            "MA Starc": ("Starc", "Mitchell Starc"),
        }
    )
)

# every arrow, read from the real graph; a branch is drawn dotted
arrows = []
for edge in graph.get_graph().edges:
    arrows.append((edge.source, edge.target, edge.conditional))

agents = []
for agent in nodes.AGENTS:
    agents.append(agent.__name__)

# the parts of the v4 figure, in the order a question meets them
PARTS = [
    ["__start__", "n1_video_input", "n2_n5_shot_classifier", "confidence_gate", "n2b_pose_estimator"],
    ["n3_question_reader", "n6_shot_dna"],
    ["n7a_evidence_retrieval", "n7b_context_retrieval", "n8_descriptive_engine", "n9_predictive_engine"],
    ["n10_supervisor_router", *agents],
    ["faith_gate", "n12_report_synthesis", "n13_final_output", "__end__"],
]

# a node the parts forget would never be drawn: say so
placed = []
for part in PARTS:
    placed += part
for name in graph.get_graph().nodes:
    if name not in placed:
        print("not in any part:", name)


def show_part(number):
    """Draw parts 1 to number; the newest part in yellow, a branch dotted."""
    shown = []
    for part in PARTS[:number]:
        shown += part
    lines = ["graph TD"]
    for source, target, branch in arrows:
        if source in shown and target in shown:
            if branch:
                lines.append(f"    {source} -.-> {target}")
            else:
                lines.append(f"    {source} --> {target}")
    for name in PARTS[number - 1]:
        lines.append(f"    class {name} new")
    lines.append("    classDef new fill:#ffd966,stroke:#b8860b,color:#000")
    display(Markdown("```mermaid\n" + "\n".join(lines) + "\n```"))

## Part 1: the clip

The dotted arrows are a branch (Day 56): the video nodes run only when a clip comes with the question. N1 reads the frames. The shot classifier (N2 to N5, your CricShot model) names the shot, the confidence gate checks how sure it is, and the pose estimator (N2b) finds the body's joints, all from the same frames.

In [ ]:
show_part(1)

## Part 2: the question

N3 reads the question, every time: by simple rules since Day 56, by a language model from Day 58. Shot DNA (N6) waits for whoever is still coming: the shot, the body and the question with a clip; the question alone without one.

In [ ]:
show_part(2)

## Part 3: the evidence and the engines

N7a fetches the balls and N7b the context (career, venue). Then the engines compute: N8 counts the past, N9 simulates the future. Both engines wait for both retrievers.

In [ ]:
show_part(3)

## Part 4: the router and the agents

N10 waits for both engines and chooses at most four agents. Since Day 73 only those run, side by side. With nobody chosen, no agent runs, and the question goes straight to the answer (the dotted arrow to N13 in Part 5).

In [ ]:
show_part(4)

## Part 5: the checks and the answer: the whole graph

The Faith gate checks each agent's last try. The dotted arrows back to the agents are the bounded loop (Day 74): only those that failed go back, at most twice. N12 writes the report from what passed, and N13 shows the answer.

In [ ]:
show_part(5)

## The branch: with a clip, and without

The same question twice. Without a clip, the four video nodes never run.

In [ ]:
text = "Kohli out within 6 balls in the T20 powerplay against Bumrah?"

for clip in ("clip.mp4", None):
    final = graph.invoke({"question": text, "clip": clip}, context=CONTEXT)
    skipped = []
    for node in nodes.EVERY_NODE:
        if node.__name__ not in final["visited"]:
            skipped.append(node.__name__)
    print("clip:", clip, "|", len(final["visited"]), "nodes ran | skipped:", skipped)

## One question, step by step

Each line is one node finishing, and what it wrote in the notebook (the state). Look at N3: it writes the subject it read. Nodes that do not wait for each other run side by side.

In [ ]:
question = {"question": text, "clip": None}

for chunk in graph.stream(question, stream_mode="updates", context=CONTEXT):
    for node, update in chunk.items():
        print(node, "->", update)

## Try your own question

Change the text and run it. The rules know only the three players above, the shots, the phases, pace and spin, the formats, "N balls" and "women".

In [ ]:
mine = "Kohli against Bumrah or Starc in ODIs, cover drive or pull?"

final = graph.invoke({"question": mine, "clip": None}, context=CONTEXT)
print(final["subject"])
print(final["answer"])

## Day 60: the Shot DNA, the balls and the context, from the real database

The same question, now with the real database (read-only) and the real registers. N6 says why nothing is seen yet (no clip). N7a fetches the balls of each side, each filed under its own subject, or says why it fetched none (a shot cannot be picked yet). N7b says who the people are, in the registers' words, never a statistic.

In [ ]:
import dataclasses

import duckdb

from cricketmind_v3.infrastructure.duckdb_repo import DuckDBDeliveryRepository
from cricketmind_v3.pipeline.grounds import load_grounds
from cricketmind_v3.pipeline.players import load_players
from cricketmind_v3.pipeline.teams import load_teams

db = duckdb.connect("../data/cricket.duckdb", read_only=True)  # never changes the data
real = dataclasses.replace(
    CONTEXT,
    deliveries=DuckDBDeliveryRepository(db),
    players=tuple(load_players(db)),
    grounds=tuple(load_grounds(db)),
    teams=tuple(load_teams(db)),
)

asked = "Kohli against Bumrah or Starc in ODIs?"
final = graph.invoke({"question": asked, "clip": None}, context=real)
print("N6:", final["shot_dna"].empty_because)
for found in final["evidence"]:
    print("N7a:", found.subject.player, "vs", found.subject.bowler, "|", found.count, "balls |", found.none_because)
print("N7b:", final["context"].people)
db.close()

## Day 61: the engines write the fact surface

The engines now answer inside the graph. N8 counts the past (k taken out) and N9 simulates the future (k kept), side by side, one question to the database at a time. Every measure on the menu that fits the question is asked. Each line is one row of the surface: what was asked, the number, how many balls, and the engine that made it. This is the list agents will read; they never ask an engine themselves.

In [ ]:
from cricketmind_v3.application.fact_surface import surface
from cricketmind_v3.infrastructure.duckdb_engines import engines_on

db = duckdb.connect("../data/cricket.duckdb", read_only=True)  # never changes the data
# N7a and the engines each on their own cursor, as a query on one connection can swap rows
real = dataclasses.replace(
    CONTEXT,
    deliveries=DuckDBDeliveryRepository(db.cursor()),
    engines=engines_on(db.cursor()),
)

asked = "Kohli out within 6 balls in the T20 powerplay against pace?"
final = graph.invoke({"question": asked, "clip": None}, context=real)
for row in final["answers"]:
    if row.fact is None:
        print(row.metric, "| no answer")
    else:
        print(row.metric, "|", round(row.fact.value, 3), "| n =", row.fact.n, "|", row.fact.receipt.engine)
print("the surface:", surface(final["answers"]))
db.close()

## Day 62: which agents can answer

N10 now reads the fact surface. An agent can answer only if the question has a real number for at least one of its measures; `capable` lists each one with those measures. A rule goes to the Explainer alone. The router still sends every agent for now; from Day 73 it sends only the ones it chooses from this list.

In [ ]:
db = duckdb.connect("../data/cricket.duckdb", read_only=True)  # never changes the data
real = dataclasses.replace(
    CONTEXT,
    deliveries=DuckDBDeliveryRepository(db.cursor()),
    engines=engines_on(db.cursor()),
)

for asked in (
    "Kohli out within 6 balls in the T20 powerplay against pace?",
    "Kohli: cover drive or pull?",  # nobody yet: shots have no labels until Day 139
    "What is LBW?",
):
    final = graph.invoke({"question": asked, "clip": None}, context=real)
    print(asked)
    if not final.get("capable"):
        print("   nobody can answer")
    for agent, measures in final.get("capable", {}).items():
        print("  ", agent, measures)
db.close()

## Day 63: who should answer

Two stages. A proposal says who should answer, best first; today it comes from words in the question. Then fixed rules decide: only agents with real numbers, at most four, the default pair when nothing else is left, and nobody when no agent has a number. `routing.no_number` writes down who was wanted but has no number.

In [ ]:
db = duckdb.connect("../data/cricket.duckdb", read_only=True)  # never changes the data
real = dataclasses.replace(
    CONTEXT,
    deliveries=DuckDBDeliveryRepository(db.cursor()),
    engines=engines_on(db.cursor()),
)

for asked in (
    "Kohli out within 6 balls in the T20 powerplay against pace?",
    "Should I pick Kohli as my fantasy captain in T20s?",
    "Kohli: cover drive or pull?",
    "What is LBW?",
):
    final = graph.invoke({"question": asked, "clip": None}, context=real)
    routing = final["routing"]
    print(asked)
    print("   can answer:", len(final["capable"]), "| chosen:", routing.chosen, "| how:", routing.how)
    if routing.no_number:
        print("   wanted, but no number:", routing.no_number)
db.close()

## Day 64: the model proposes, and the rules still decide

With a model, it reads the question and an evidence summary, and proposes. Here a **scripted fake** stands in for the model, so we can see exactly what a real one would be told. The fake asks for the medical team too, who have no number, and the rules remove them. To try the real model, when the GPU is free: `uv run python scripts/route_questions.py ../data/cricket.duckdb qwen2.5:7b`.

In [ ]:
from cricketmind_v3.application.ports import FakeLLM
from cricketmind_v3.application.router import SYSTEM, route

db = duckdb.connect("../data/cricket.duckdb", read_only=True)  # never changes the data
real = dataclasses.replace(
    CONTEXT,
    deliveries=DuckDBDeliveryRepository(db.cursor()),
    engines=engines_on(db.cursor()),
)
asked = "Would you start Kohli in my T20 league team?"
final = graph.invoke({"question": asked, "clip": None}, context=real)
db.close()

scripted = '{"reason": "a question from a fan", "agents": ["N11e", "N11k"]}'
fake = FakeLLM([scripted])
routing = route(asked, final["capable"], final["evidence"], fake)

print("THE MODEL IS TOLD:")
print(SYSTEM)
print()
print(fake.asked[0][1])
print()
print("THE FAKE PROPOSES:", routing.proposed)
print("THE RULES CHOOSE: ", routing.chosen, "| no number:", routing.no_number)

## Day 65: what an agent is told

One small file for each agent, in `backend/prompts/`, says who it works for and how it sounds. One shared file teaches the checks in plain words. The list of measures comes from the roster, never from a file, and only those with numbers for this question are shown. No model reads these until Day 67.

In [ ]:
from cricketmind_v3.domain.agent_spec import system_prompt
from cricketmind_v3.infrastructure.prompt_files import read_rules, read_specs

specs = read_specs()
print(len(specs), "agents have a prompt file\n")
print(system_prompt(specs["N11b"], read_rules(), ("strike_rate", "p_dismissal_within_k")))

## Day 66: the model writes only the words

The model fills one box: the sentence, with a slot like `{c1}` where each number goes. The code makes the claims from what the engines said. **The numbers here are made up, and a scripted fake stands in for the model:** no real answer comes before the exam is locked (Day 67).

In [ ]:
import json

from cricketmind_v3.application.agent_factory import write_finding
from cricketmind_v3.application.ports import FakeFactSource, FakeLLM, FakeLLMByWho
from cricketmind_v3.application.reader import QuestionForm
from cricketmind_v3.domain.agent_spec import agent_prompt, offered_claims
from cricketmind_v3.domain.facts import Answer, Fact
from cricketmind_v3.domain.names import Player
from cricketmind_v3.domain.render import render
from cricketmind_v3.domain.subject import CricketSubject
from cricketmind_v3.infrastructure.prompt_files import read_rules, read_specs

specs = read_specs()
rules = read_rules()
asked = "Kohli's strike rate in T20s?"
kohli = CricketSubject(player="V Kohli", match_format="T20")
made_up = [
    Answer(metric="strike_rate", subject=kohli, fact=Fact(value=135.1, n=9841)),
    Answer(metric="sr_vs_average", subject=kohli, fact=Fact(value=1.0, n=9841, interval=(-2.0, 4.0))),
]

print("THE MODEL IS TOLD:")
print(agent_prompt(asked, offered_claims("N11b", made_up)))
print()
scripted = FakeLLM([json.dumps({"template": "Kohli returns {c1}."})])
finding = write_finding("N11b", asked, made_up, scripted, specs["N11b"], rules)
print("IT WRITES:   ", finding.template)
print("THE CODE KEEPS:", finding.claims[0].metric, finding.claims[0].value, "from", finding.claims[0].source)
print("THE READER SEES:", render(finding))

## Days 68-71: twelve agents, one factory

What differs is the voice and the measures. A measure on nobody's list, or with no engine, stops the graph being built at all (I2). The medical agent's words must follow its flag: `overworked` only with a spike, and never `safe`.

In [ ]:
from cricketmind_v3.application.faith_check import faith_check
from cricketmind_v3.domain.faith.gate import Context
from cricketmind_v3.domain.roster import NAMES, ROSTER

for agent, measures in ROSTER.items():
    print(agent, NAMES[agent], "->", len(measures), "measures")

workload = CricketSubject(bowler="JJ Bumrah", date="2025-01-04")
usual = Fact(value=1.32, n=626)  # made up here; the real one is N-018
flag = [Answer(metric="workload_ratio_flag", subject=workload, fact=usual)]
source = FakeFactSource()
source.add("workload_ratio_flag", workload, usual)
gate = Context(allowed=ROSTER, names=CONTEXT.names)
print()
for words in ("Bumrah shows {c1}.", "Bumrah looks overworked: {c1}.", "Bumrah shows {c1}, so he is safe."):
    llm = FakeLLM([json.dumps({"template": words})])
    finding = write_finding("N11k", "Bumrah's workload?", flag, llm, specs["N11k"], rules)
    wrong = []
    for failure in faith_check(finding, source, gate):
        wrong.append(failure.reason)
    print(words, "->", wrong or render(finding))

## Day 72: the Explainer

Words only. It may name no player and type no digit, and its text always carries its label. `ball` and `chance` are also players in the register, so a name needs its capital (D-089).

In [ ]:
for words in (
    "A ball is called wide when the batter has no chance to reach it.",
    "An over has 6 balls.",
    "LBW is how Kohli was out.",
):
    llm = FakeLLM([json.dumps({"template": words})])
    finding = write_finding("N11m", "What is LBW?", [], llm, specs["N11m"], rules)
    wrong = []
    for failure in faith_check(finding, FakeFactSource(), gate):
        wrong.append(failure.check + ": " + failure.reason)
    print(words, "->", wrong or "passes")

## Days 73-75: the whole road, with a scripted model

The first agent lies once, by typing a number; the gate sends it back with the reason, and it fixes it. The second lies every time, and is withheld after three tries. The numbers and the words are all made up.

In [ ]:
from dataclasses import replace

players = (
    Player(person_id="ba607b88", name="V Kohli", matches=600),
    Player(person_id="462411b3", name="JJ Bumrah", matches=400),
)


def reading(**fields):
    reply = dict.fromkeys(QuestionForm.model_fields)
    reply.update(kind="stats", player="Kohli", match_format="T20")
    reply.update(fields)
    return json.dumps(reply)


def run_scripted(agents, chosen, question, facts, report=()):
    # a fake engine and a fake model: nothing here is about the real Kohli
    source = FakeFactSource()
    for metric, subject, fact in facts:
        source.add(metric, subject, fact)
    replies = {
        "You read cricket questions": [reading()],
        "You send cricket questions": [json.dumps({"reason": "scripted", "agents": chosen})],
        "You join specialists": list(report),
    }
    for name, words in agents.items():
        replies[name] = []
        for each in words:
            replies[name].append(json.dumps({"template": each}))
    scripted_context = replace(
        CONTEXT, llm=FakeLLMByWho(replies), players=players, engines=source, specs=specs, rules=rules
    )
    return graph.invoke({"question": question, "clip": None}, context=scripted_context)


rate = ("strike_rate", kohli, Fact(value=135.1, n=9841))
final = run_scripted(
    {
        "Tactical Analysis": ["Kohli returns {c1}, close to 150.", "Kohli returns {c1}."],
        "Personal Performance": ["Kohli is outstanding: {c1}."] * 3,
    },
    ["N11b", "N11c"],
    asked,
    [rate],
)
print("Tactical Analysis ran", final["visited"].count("n11b_tactical_analysis"), "times")
print("Personal Performance ran", final["visited"].count("n11c_personal_performance"), "times")
print("the gate checked", final["checks_run"], "times")
print()
print(final["answer"])

## Day 76: a judgment hidden in a question

`Why is Kohli bad...` holds a judgment. The same test as C7's: a checked comparison whose whole range lies on the word's side of zero. The answer says it first. The numbers are made up: try the other two facts.

In [ ]:
above = Fact(value=12.0, n=9841, interval=(8.0, 16.0))
level = Fact(value=1.0, n=9841, interval=(-2.0, 4.0))
thin = Fact(value=-40.0, n=6, interval=(-90.0, -5.0))

for fact in (above, level, thin):
    final = run_scripted(
        {"Tactical Analysis": ["Kohli returns {c1}."]},
        ["N11b"],
        "Why is Kohli so bad in T20s?",
        [rate, ("sr_vs_average", kohli, fact)],
    )
    print(final["answer"].splitlines()[1])

## Day 79: what a question can cost at worst

Every loop has a limit, so the worst case is a sum anyone can check. Here four agents lie every time, and we count.

In [ ]:
from cricketmind_v3.domain import cost

print("the bound:", cost.text_supersteps(), "supersteps,", cost.agent_runs(), "agent runs,", cost.model_calls(), "model calls")

four = ["N11a", "N11b", "N11c", "N11d"]
agents = {}
for agent in four:
    agents[NAMES[agent]] = ["Kohli returns {c1}, close to 150."] * 3
final = run_scripted(agents, four, asked, [rate])
runs = 0
for name in final["visited"]:
    if name.startswith("n11"):
        runs += 1
print("this run:", runs, "agent runs, and", len(final["withheld"]), "withheld")